In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
protein = "VCAM1"

In [ ]:
vgat_geph_results_file = f"/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/{protein}/{protein}-LacZ_VGAT-GEPH_output_data/metric_results.csv"
vgat_geph_results = pd.read_csv(vgat_geph_results_file)
vgat_geph_results

In [ ]:
vgat_geph_results["target"] = vgat_geph_results["gRNA"].str.cat(vgat_geph_results[["Brain"]], sep = "_")

In [ ]:
vgat_geph_results

In [ ]:
df2_grouped_brain = vgat_geph_results.groupby(["target", "hippocampal_layer"], as_index=False)[["local_peak_colocalized_spots",
                                        "overlap_coeff",
                                        "overlap_um2",
                                        "pearson_cor",
                                        "presynapse_image_mfi",
                                        "postsynapse_image_mfi",
                                        "pre_puncta_density_per_100_um2",
                                        "post_puncta_density_per_100_um2",
                                        "pre_staining_area_um2",
                                        "post_staining_area_um2",
                                        "pre_mean_puncta_size_um2",
                                        "post_mean_puncta_size_um2"]].mean()

In [ ]:
df_grouped_section = vgat_geph_results[["target",
                                        "hippocampal_layer",
                                        "img_filename",
                                        "local_peak_colocalized_spots",
                                        "overlap_coeff",
                                        "overlap_um2",
                                        "pearson_cor",
                                        "presynapse_image_mfi",
                                        "postsynapse_image_mfi",
                                        "pre_puncta_density_per_100_um2",
                                        "post_puncta_density_per_100_um2",
                                        "pre_staining_area_um2",
                                        "post_staining_area_um2",
                                        "pre_mean_puncta_size_um2",
                                        "post_mean_puncta_size_um2"]]
final_df = df_grouped_section.reset_index()

In [ ]:
# to long format
df_melted = df2_grouped_brain.melt(id_vars=['target', 'hippocampal_layer'], var_name='metric', value_name='value')

# create new column based on hippocampal layer and metric
df_melted['hippocampal_layer_metric'] = df_melted['hippocampal_layer'] + '_' + df_melted['metric']

# pivot the dataframe back
df_pivot = df_melted.pivot(index='target', columns='hippocampal_layer_metric', values='value')

final_df = df_pivot.reset_index()

In [ ]:
features = list(final_df.columns)
columns_to_remove = ["index", "target", "hippocampal_layer", "img_filename"]

for column in columns_to_remove:
    if column in features:  # Check if the column exists in the list
        features.remove(column)

print(features)

In [ ]:
# separating out the features
X = final_df.loc[:,features].values
# separating out the target
Y = final_df.loc[:,["target"]].values

In [ ]:
scaled_data = StandardScaler().fit_transform(X)

In [ ]:
pca = PCA(n_components=2)
principal_components = pca.fit_transform(scaled_data)

In [ ]:
pc_df = pd.DataFrame(data = principal_components, columns = ['principal component 1', 'principal component 2'])
pc_df.head(5)

In [ ]:
pca.explained_variance_ratio_

In [ ]:
pca_df = pd.concat([pc_df, final_df[['target']]], axis = 1)
pca_df.head(10)

In [ ]:
# Step 1: Create a new column to group targets into "LacZ-gRNA" and "VCAM1-gRNA"
pca_df['group'] = pca_df['target'].apply(lambda x: 'LacZ-gRNA' if 'LacZ' in x else 'VCAM1-gRNA')

# Step 2: Create a scatter plot using Seaborn
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=pca_df,
    x='principal component 1', 
    y='principal component 2', 
    hue='group',  # Use the new 'group' column for coloring
    palette='Set1',  # Set1 is a good palette for distinct colors
    s=100  # Marker size
)

# Add title and labels
plt.title('PCA Scatter Plot: LacZ-gRNA vs VCAM1-gRNA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

# Show the plot
plt.show()

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

iris.frame

In [ ]:
import umap

In [ ]:
reducer = umap.UMAP()

In [ ]:
embedding = reducer.fit_transform(scaled_data)
embedding.shape

In [ ]:
plt.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=[sns.color_palette()[x] for x in final_df.target.map({"LacZ-gRNA_Brain-4":0, 
                                                            "LacZ-gRNA_Brain-4-2":1, 
                                                            "LacZ-gRNA_Brain-5":0,
                                                            "LacZ-gRNA_Brain-7":0,
                                                            "VCAM1-gRNA_Brain-4":1,
                                                            "VCAM1-gRNA_Brain-4-2":1,
                                                            "VCAM1-gRNA_Brain-5":1,
                                                            "VCAM1-gRNA_Brain-7":1})])
plt.gca().set_aspect('equal', 'datalim')
plt.title('UMAP ', fontsize=24);

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Corrected label mapping
label_map = {
    "CA1 SLM_VCAM1-gRNA": 0, 
    "CA1 SLM_LacZ-gRNA": 1, 
    "CA1 SR_VCAM1-gRNA": 2, 
    "CA1 SR_LacZ-gRNA": 3,
    "CA1 SO_VCAM1-gRNA": 4, 
    "CA1 SO_LacZ-gRNA": 5, 
    "CA3 SO_VCAM1-gRNA": 6, 
    "CA3 SO_LacZ-gRNA": 7,
    "CA3 SL_VCAM1-gRNA": 8, 
    "CA3 SL_LacZ-gRNA": 9, 
    "CA3 SR_VCAM1-gRNA": 10, 
    "CA3 SR_LacZ-gRNA": 11,
    "DG Hilus_VCAM1-gRNA": 12, 
    "DG Hilus_LacZ-gRNA": 13,
    "DG ML_VCAM1-gRNA": 14, 
    "DG ML_LacZ-gRNA": 15
}

label_map_gRNA = {
    "VCAM1-gRNA": 0, 
    "LacZ-gRNA": 1
}

# Generate base colors for each layer
base_colors = sns.color_palette("husl", n_colors=2)

# Create color palette with slight variations for VCAM1 and LacZ
colors = []
for base_color in base_colors:
    # Slightly darker color for VCAM1-gRNA
    colors.append(tuple(max(0, c - 0.1) for c in base_color))
    # Slightly lighter color for LacZ-gRNA
    colors.append(tuple(min(1, c + 0.1) for c in base_color))

# Map the labels to colors
color_values = [colors[x] for x in vglut1_psd95_results.gRNA.map(label_map_gRNA)]

# Create a larger plot with smaller dots
plt.figure(figsize=(8, 8))  # Increase the plot size

# Plot the scatter points
scatter = plt.scatter(
    embedding[:, 0],  # X-axis values
    embedding[:, 1],  # Y-axis values
    c=color_values,   # Colors based on mapped labels
    s=15             # Dot size, reduced for smaller dots
)

plt.gca().set_aspect('equal', 'datalim')
plt.title('UMAP Visualization', fontsize=24)

# Create a legend manually
handles = [plt.Line2D([0], [0], marker='o', color='w', label=label, 
                      markerfacecolor=colors[i], markersize=10) 
           for label, i in label_map_gRNA.items()]
plt.legend(handles=handles, loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)

plt.tight_layout()
plt.show()

